# 작은 디코더를 학습하고, 결과가 잘못될 조건을 찾아보세요

현재 기호만 보게 해도 디코더가 반복 패턴을 복사할 수 있을까요? [저장된 테스트 손실을 비교해 보세요](#cell-id=compare-results). `test_nats`는 앞 기호까지 볼 수 있을 때, `self_only_test_nats`는 현재 기호만 볼 때의 손실입니다.

한 커널에서 코드 셀을 차례로 실행했습니다(Python 3.12.14 · PyTorch 2.15.0+cpu · CPU). 마지막 셀에 표시된 경로는 이 실행에 사용한 임시 폴더입니다.

[직접 실험하기](https://sungeuns.github.io/foundation-model-engineering/ko/chapter-7/tiny-decoder-lab/) · [Foundation Model Engineering 저장소](https://github.com/sungeuns/foundation-model-engineering)

In [1]:
# Replace the empty strings with your predictions before running the experiment.
predictions = {
    "earlier_logits_change_when_future_tokens_change": "",
    "self_only_attention_helps_or_hurts": "",
    "resumed_training_matches_uninterrupted_training": "",
}
seed, steps = 7, 160


## 준비와 실습 범위

Python 3.12 이상과 PyTorch가 설치된 Jupyter 커널을 사용하세요. 아래 import가 실패하면 [공식 설치 안내](https://pytorch.org/get-started/locally/)에 따라 **해당 커널의 환경** 에 PyTorch를 설치하세요. 이 노트북은 패키지를 자동으로 설치하거나 갱신하지 않습니다. 2026-11-18에 Python 3.12, PyTorch 2.15.0+cpu에서 검증했습니다. 다른 호환 환경에서도 실행할 수 있지만 버전·하드웨어를 바꿨을 때 수치 경로가 같다고 보장하지는 않습니다.

고정 레시피는 한 층, 폭 32, 어텐션 헤드 두 개, FP32 CPU, 배치 16, AdamW입니다. 학습률 스케일은 0.003, betas는 0.9/0.95, epsilon은 1e-8, 감쇠는 0.01, 기울기 클리핑은 1.0입니다. 열 번의 갱신 동안 적용하는 warmup 계수와 cosine 계수를 곱하며 정확한 식은 소스에 있습니다. 처음 네 기호가 프롬프트이고 이어지는 13개 기호에 손실을 줍니다. 순환 이동한 패턴을 같은 집단으로 묶은 뒤 train/validation/test를 128/16/16개 집단으로 나눕니다.

매개변수·기울기·Adam moment를 합하면 약 158.8 KiB이며 활성화, 임시 버퍼, 런타임, 체크포인트와 평가는 별도입니다. 여유 메모리 1 GiB는 계획치이며 측정한 최대 사용량이 아닙니다. 샤딩, 혼합 정밀도, chat template, 패딩, 원격 복구와 분산 실행은 없습니다. 처음에는 원래 레시피로 실행하세요. Test 결과를 보고 설정을 고른 뒤 새 일반화 주장을 하려면 새로운 보류셋이 필요합니다.


In [2]:
import hashlib
import importlib.util
import json
import math
from pathlib import Path
import sys
import tempfile
import time
from zipfile import ZIP_DEFLATED, ZipFile

if sys.version_info < (3, 12):
    raise RuntimeError("Use a Python 3.12+ kernel for this teaching lab.")
import torch

print({"python": sys.version.split()[0], "torch": str(torch.__version__), "device": "cpu"})


{'python': '3.12.14', 'torch': '2.15.0+cpu', 'device': 'cpu'}


## 실습 소스 읽기

다음 셀에는 이번 실행에 사용한 `public/labs/causal_lab.py` 전체가 들어 있습니다. nbviewer에서는 소스가 처음부터 끝까지 표시됩니다. 이어지는 준비 셀은 SHA-256으로 소스를 확인한 뒤 불러옵니다.

In [3]:
SOURCE = r'''"""CPU teaching lab: causal copying, an attention ablation, and exact resume.

Python 3.12+, PyTorch 2.15 CPU. No downloads, credentials, or external data.
Run: python causal_lab.py --output /tmp/causal-lab --steps 160 --seed 7
Not a language model benchmark or a distributed training implementation.
"""
import argparse
import copy
import hashlib
import json
import math
from pathlib import Path
import random
import time

import torch
from torch import nn
from torch.nn import functional as F


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def corpus():
    # Rotations of a motif are one semantic group. Split groups BEFORE expansion.
    rng, groups = random.Random(20261117), set()
    while len(groups) < 160:
        motif = tuple(rng.sample(range(1, 17), 4))
        groups.add(min(motif[i:] + motif[:i] for i in range(4)))
    groups = sorted(groups, key=digest)
    splits = {"train": groups[:128], "validation": groups[128:144], "test": groups[144:]}
    manifest = {"generator": "distinct-4-cycle-v1", "seed": 20261117,
                "tokenizer": "integer-identity-v1; ids 1..16; 0 unused",
                "provenance": "locally generated synthetic symbols; no personal data",
                "filter": "four distinct symbols; canonical cyclic rotation",
                "mixture": {"distinct-4-cycle": 1.0}, "groups": splits}
    tensors = {name: torch.tensor([(m * 5)[:17] for m in motifs])
               for name, motifs in splits.items()}
    return tensors, manifest


class Decoder(nn.Module):
    def __init__(self, attention=True):
        super().__init__()
        self.tokens = nn.Embedding(17, 32)
        self.positions = nn.Embedding(16, 32)
        self.norm1, self.norm2 = nn.LayerNorm(32), nn.LayerNorm(32)
        self.attention = nn.MultiheadAttention(32, 2, dropout=0.0, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(32, 64), nn.GELU(), nn.Linear(64, 32))
        self.output = nn.Linear(32, 17)
        # MultiheadAttention boolean masks use True for BLOCKED positions.
        mask = torch.triu(torch.ones(16, 16, dtype=torch.bool), diagonal=1)
        if not attention:
            mask = ~torch.eye(16, dtype=torch.bool)  # self-only ablation
        self.register_buffer("mask", mask)

    def forward(self, ids):
        length = ids.shape[1]
        x = self.tokens(ids) + self.positions(torch.arange(length))
        h = self.norm1(x)
        x = x + self.attention(h, h, h, attn_mask=self.mask[:length, :length],
                               need_weights=False)[0]
        x = x + self.ffn(self.norm2(x))
        return self.output(x)


def loss_sum(model, batch):
    logits = model(batch[:, :-1])  # [B, 16, 17]
    # Four observed symbols form the prompt; score only the 13 repeat targets.
    logits, labels = logits[:, 3:], batch[:, 4:]
    return F.cross_entropy(logits.reshape(-1, 17), labels.reshape(-1), reduction="sum"), labels.numel()


@torch.no_grad()
def evaluate(model, batch):
    model.eval()
    total, count = loss_sum(model, batch)
    return float(total / count)


@torch.no_grad()
def generate(model, prompt, continuation=13):
    """Greedy continuation sees its own predictions, never future targets."""
    model.eval()
    sequence = prompt.clone()
    for _ in range(continuation):
        prediction = model(sequence)[:, -1].argmax(dim=-1, keepdim=True)
        sequence = torch.cat([sequence, prediction], dim=1)
    return sequence


def setup(seed, attention=True):
    torch.manual_seed(seed)
    model = Decoder(attention)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, betas=(0.9, 0.95),
                                  eps=1e-8, weight_decay=0.01)
    sampler = torch.Generator().manual_seed(seed + 1000)
    return model, optimizer, sampler


def advance(model, optimizer, sampler, data, start, end, total_steps):
    model.train()
    trace = []
    for step in range(start, end):
        ids = torch.randint(len(data), (16,), generator=sampler)
        # Schedule uses the FINAL planned budget, including before interruption.
        lr = 0.003 * min((step + 1) / 10, 1.0) * (0.1 + 0.9 * (1 + math.cos(math.pi * step / total_steps)) / 2)
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)
        total, count = loss_sum(model, data[ids])
        loss = total / count
        if not torch.isfinite(loss):
            raise RuntimeError("Abort: nonfinite loss")
        loss.backward()
        norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        optimizer.step()
        trace.append({"step": step + 1, "loss_nats_per_target": float(loss.detach()),
                      "gradient_norm_before_clip": float(norm), "lr": lr,
                      "batch_ids": ids.tolist(), "target_tokens": count})
    return trace


def save_checkpoint(path, model, optimizer, sampler, step, config, manifest):
    state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
             "torch_rng": torch.get_rng_state(), "sampler_rng": sampler.get_state(),
             "step": step, "target_tokens": step * 16 * 13,
             "data_cursor": "replacement sampler RNG plus global step",
             "config": config, "manifest_hash": digest(manifest),
             "config_hash": digest(config), "scheduler": "formula in advance; step/steps",
             "precision": "CPU float32; no scaler; world size 1"}
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    temporary.replace(path)
    marker = path.with_suffix(".sha256")
    marker.write_text(hashlib.sha256(path.read_bytes()).hexdigest())


def load_checkpoint(path, config, manifest):
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != path.with_suffix(".sha256").read_text():
        raise ValueError("Incomplete or corrupt checkpoint")
    state = torch.load(path, weights_only=True)
    if state["config_hash"] != digest(config) or state["manifest_hash"] != digest(manifest):
        raise ValueError("Configuration or corpus changed")
    model, optimizer, sampler = setup(config["seed"])
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    sampler.set_state(state["sampler_rng"])
    torch.set_rng_state(state["torch_rng"])
    return model, optimizer, sampler, state["step"]


def run(output, steps=160, seed=7):
    if steps < 20:
        raise ValueError("Use at least 20 steps; learning quality is budget-dependent")
    output.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    data, manifest = corpus()
    config = {"steps": steps, "seed": seed, "batch": 16, "width": 32, "heads": 2,
              "layers": 1, "dtype": "float32", "device": "cpu",
              "code_hash": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
              "torch_version": str(torch.__version__)}
    (output / "manifest.json").write_text(json.dumps(manifest, indent=2))
    model, optimizer, sampler = setup(seed)
    changed = data["train"][:2, :-1].clone()
    original = model(changed).detach()
    changed[:, 8:] = 1
    assert torch.equal(original[:, :8], model(changed).detach()[:, :8]), "Future leakage"
    initial = evaluate(model, data["validation"])
    start_time = time.perf_counter()
    first = advance(model, optimizer, sampler, data["train"], 0, steps // 2, steps)
    checkpoint = output / "checkpoint.pt"
    save_checkpoint(checkpoint, model, optimizer, sampler, steps // 2, config, manifest)
    tail = advance(model, optimizer, sampler, data["train"], steps // 2, steps, steps)
    expected = copy.deepcopy(model.state_dict())
    resumed, resumed_optimizer, resumed_sampler, cursor = load_checkpoint(checkpoint, config, manifest)
    replay = advance(resumed, resumed_optimizer, resumed_sampler, data["train"], cursor, steps, steps)
    assert tail == replay, "Resume changed batches, loss, or gradients"
    assert all(torch.equal(value, resumed.state_dict()[key]) for key, value in expected.items())
    elapsed = time.perf_counter() - start_time
    ablated, ablated_optimizer, ablated_sampler = setup(seed, attention=False)
    advance(ablated, ablated_optimizer, ablated_sampler, data["train"], 0, steps, steps)
    generated = generate(model, data["test"][:, :4])
    correct = generated[:, 4:] == data["test"][:, 4:]
    result = {"config": config, "parameters": sum(p.numel() for p in model.parameters()),
              "initial_validation_nats": initial,
              "validation_nats": evaluate(model, data["validation"]),
              "test_nats": evaluate(model, data["test"]),
              "self_only_test_nats": evaluate(ablated, data["test"]),
              "greedy_test_token_accuracy": float(correct.float().mean()),
              "greedy_test_sequence_accuracy": float(correct.all(dim=1).float().mean()),
              "example": {"prompt": data["test"][0, :4].tolist(),
                          "generated": generated[0, 4:].tolist(),
                          "expected": data["test"][0, 4:].tolist()},
              "exact_resume": True, "causal_invariance": True,
              "train_and_replay_seconds": elapsed,
              "train_and_replay_target_tokens_per_second": (steps + steps - steps // 2) * 208 / elapsed,
              "nonfinite_count": 0, "mfu": None}
    (output / "trace.json").write_text(json.dumps(first + tail, indent=2))
    (output / "result.json").write_text(json.dumps(result, indent=2))
    return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--steps", type=int, default=160)
    parser.add_argument("--seed", type=int, default=7)
    args = parser.parse_args()
    print(json.dumps(run(args.output, args.steps, args.seed), indent=2))
'''


In [4]:
EXPECTED_SOURCE_SHA256 = "d3488b924a81a2cc50a381583903e8756c7b76846d1221cabd63e490bba1ba0c"
assert hashlib.sha256(SOURCE.encode("utf-8")).hexdigest() == EXPECTED_SOURCE_SHA256
run_root = Path(tempfile.mkdtemp(prefix="fme-tiny-decoder-", dir=Path.cwd()))
output = run_root / "outputs"
output.mkdir()
source_path = output / "causal_lab.py"
source_path.write_bytes(SOURCE.encode("utf-8"))
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
spec = importlib.util.spec_from_file_location("fme_causal_lab", source_path)
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
print({"source_sha256": EXPECTED_SOURCE_SHA256, "output_directory": str(output)})


{'source_sha256': 'd3488b924a81a2cc50a381583903e8756c7b76846d1221cabd63e490bba1ba0c', 'output_directory': '/tmp/fme-notebook-smoke-0_9hmung/fme-tiny-decoder-66wgggj2/outputs'}


## 원래 실험 실행하기

인과 모델을 학습하고 중간 체크포인트를 저장·복원한 뒤 남은 갱신을 재실행합니다. 자기 위치만 보는 어텐션 모델도 학습합니다. CPU 구현은 인과성 불변 조건과 정확한 재개를 검사하며, 손실·기울기가 비유한값이거나 검사가 실패하면 중단합니다. 토큰 정규화 손실, 기울기 norm, 학습률, 배치 ID와 처리량을 기록합니다. 검증된 하드웨어 FLOP 모델이 없으므로 MFU는 0으로 채우지 않고 미보고 값으로 남깁니다.


In [5]:
started = time.perf_counter()
result = lab.run(output, steps=steps, seed=seed)
notebook_run_seconds = time.perf_counter() - started
print({"full_experiment_seconds": round(notebook_run_seconds, 3),
       "parameters": result["parameters"], "supervised_training_tokens": steps * 16 * 13})


{'full_experiment_seconds': 1.852, 'parameters': 10161, 'supervised_training_tokens': 33280}


## 먼저 유효한 결과인지 확인하기

낮은 손실만으로는 충분하지 않습니다. 아래 검사는 이 CPU 실행과 소스 버전에 대한 것이며 분산 장애 복구나 버전 간 결정성을 입증하지 않습니다. 위의 전체 실험 시간에는 ablation과 평가가 들어갑니다. 원본 실습이 따로 기록한 학습·재실행 시간에는 이 두 작업이 빠져 있습니다.


In [6]:
assert result["causal_invariance"] is True
assert result["exact_resume"] is True
assert result["nonfinite_count"] == 0
assert result["config"]["code_hash"] == EXPECTED_SOURCE_SHA256
print({key: result[key] for key in ("causal_invariance", "exact_resume", "nonfinite_count", "mfu")})


{'causal_invariance': True, 'exact_resume': True, 'nonfinite_count': 0, 'mfu': None}


## 앞 기호를 볼 수 있는 것이 얼마나 도움이 됐나요?

자기 위치만 보는 모델은 매개변수 수, 초기화, 샘플링과 갱신 예산이 같습니다. 이 모델의 test 손실을 인과 모델과 비교하세요. 모델 크기가 아니라 이용할 수 있는 문맥을 바꾼 실험입니다. 유효 기호 16개에 균등 확률을 주는 기준선의 손실은 target당 ln(16) ≈ 2.773 nats입니다.

Teacher-forced 손실을 구할 때는 앞의 정답 기호를 제공합니다. Greedy 생성에서는 모델의 예측을 다시 입력하므로 초반 오류가 다음 토큰에도 영향을 줄 수 있습니다. 전체 시퀀스 정답률은 토큰 정답률보다 엄격한 기준입니다. 합성 보류 집단에서 얻은 관측값이며 LLM 벤치마크 점수가 아닙니다.


In [7]:
measurements = {key: result[key] for key in (
    "initial_validation_nats", "validation_nats", "test_nats", "self_only_test_nats",
    "greedy_test_token_accuracy", "greedy_test_sequence_accuracy")}
measurements["uniform_valid_symbol_nats"] = math.log(16)
print(json.dumps(measurements, indent=2))
print(json.dumps(result["example"], indent=2))
trace = json.loads((output / "trace.json").read_text())
for index in (0, len(trace) // 2, len(trace) - 1):
    print({key: trace[index][key] for key in ("step", "loss_nats_per_target", "lr", "gradient_norm_before_clip")})


{
  "initial_validation_nats": 3.0856950283050537,
  "validation_nats": 0.04528987035155296,
  "test_nats": 0.050811074674129486,
  "self_only_test_nats": 2.7041945457458496,
  "greedy_test_token_accuracy": 0.9807692170143127,
  "greedy_test_sequence_accuracy": 0.9375,
  "uniform_valid_symbol_nats": 2.772588722239781
}
{
  "prompt": [
    9,
    14,
    10,
    16
  ],
  "generated": [
    9,
    14,
    10,
    16,
    9,
    14,
    10,
    16,
    9,
    14,
    10,
    16,
    9
  ],
  "expected": [
    9,
    14,
    10,
    16,
    9,
    14,
    10,
    16,
    9,
    14,
    10,
    16,
    9
  ]
}
{'step': 1, 'loss_nats_per_target': 3.1487460136413574, 'lr': 0.00030000000000000003, 'gradient_norm_before_clip': 1.0194655656814575}
{'step': 81, 'loss_nats_per_target': 0.4234147369861603, 'lr': 0.0016500000000000002, 'gradient_norm_before_clip': 0.7720319628715515}
{'step': 160, 'loss_nats_per_target': 0.020919401198625565, 'lr': 0.0003002602253492125, 'gradient_norm_before_clip'

## 가정 하나를 깨기: 미래를 보게 만들기

별도의 미학습 진단 모델에서 인과 마스크를 없앱니다. 입력의 위치 8 이후만 바꾼 뒤 그보다 앞선 logit의 변화를 측정합니다. 변화가 0이 아니면 미래 정보가 새어 들어간 것입니다. 새 학습 결과나 더 좋은 복사 모델을 얻은 것이 아니며 원래 모델과 저장 결과는 유지됩니다.


In [8]:
diagnostic, _, _ = lab.setup(seed)
diagnostic.eval()
diagnostic.mask.zero_()
data, manifest = lab.corpus()
before = data["train"][:2, :-1].clone()
after = before.clone()
after[:, 8:] = 1
with torch.no_grad():
    leak_delta = float((diagnostic(before)[:, :8] - diagnostic(after)[:, :8]).abs().max())
assert leak_delta > 1e-6, "The diagnostic did not expose leakage; inspect the intervention."
print({"unmasked_earlier_logit_max_change": leak_delta})


{'unmasked_earlier_logit_max_change': 0.1990128457546234}


## 체크포인트 사본 손상시키기

저장된 체크포인트에는 모델·옵티마이저 상태, 전역 step·토큰 수, torch·sampler RNG, 데이터 커서, 스케줄, 설정·manifest 해시가 있습니다. SHA-256 완료 표식으로 이번 실습의 우발적 손상을 탐지합니다. 신뢰할 수 없는 다운로드의 출처를 인증하는 기능은 아닙니다. 이 노트북에서 방금 만든 체크포인트만 읽고, 온전한 결과 묶음을 정상 복구 지점으로 보존하세요.


In [9]:
broken_dir = run_root / "intentional-corruption"
broken_dir.mkdir(exist_ok=True)
broken = broken_dir / "checkpoint.pt"
broken.write_bytes((output / "checkpoint.pt").read_bytes() + b"deliberate-corruption")
broken.with_suffix(".sha256").write_text((output / "checkpoint.sha256").read_text())
try:
    lab.load_checkpoint(broken, result["config"], manifest)
except ValueError as error:
    corruption_rejected = True
    print({"corruption_rejected": True, "reason": str(error)})
else:
    raise AssertionError("A corrupt checkpoint was accepted.")
# Confirm that the untouched original still restores.
lab.load_checkpoint(output / "checkpoint.pt", result["config"], manifest)


{'corruption_rejected': True, 'reason': 'Incomplete or corrupt checkpoint'}


(Decoder(
   (tokens): Embedding(17, 32)
   (positions): Embedding(16, 32)
   (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
   (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
   (attention): MultiheadAttention(
     (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
   )
   (ffn): Sequential(
     (0): Linear(in_features=32, out_features=64, bias=True)
     (1): GELU(approximate='none')
     (2): Linear(in_features=64, out_features=32, bias=True)
   )
   (output): Linear(in_features=32, out_features=17, bias=True)
 ),
 AdamW (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.95)
     capturable: False
     decoupled_weight_decay: True
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0016765054848218483
     maximize: False
     weight_decay: 0.01
 ),
 80)

## 근거를 보존한 뒤 하나씩 바꾸기

ZIP에는 실행한 원본 소스, 불변 합성 manifest, 체크포인트와 표식, 학습 trace, 결과가 들어갑니다. 일부러 손상시킨 사본은 제외합니다. 예상 결과와 함께 보존하고 **새 실행** 에서 설정을 하나씩 바꾸세요. Seed 세 개로 민감도를 볼 수는 있지만 정밀한 신뢰구간을 얻었다고 주장할 수는 없습니다. 불변 조건·무결성·재개 검사가 실패하거나 보류 결과를 설명할 수 없다면 변경한 실습 산출물을 배포하지 마세요.

아래에 ZIP 경로가 표시됩니다. 호스팅된 노트북에서는 런타임이 사라지기 전에 파일 탐색기로 내려받으세요. 인증 정보나 내려받은 데이터셋은 포함되지 않습니다.


In [10]:
evidence = {"predictions": predictions, "full_experiment_seconds": notebook_run_seconds,
            "unmasked_earlier_logit_max_change": leak_delta, "corruption_rejected": corruption_rejected}
(output / "notebook-observations.json").write_text(json.dumps(evidence, indent=2))
archive = run_root / "result-packet.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as packet:
    for name in ("causal_lab.py", "manifest.json", "checkpoint.pt", "checkpoint.sha256",
                 "trace.json", "result.json", "notebook-observations.json"):
        packet.write(output / name, arcname=f"outputs/{name}")
print({"download_packet": str(archive)})


{'download_packet': '/tmp/fme-notebook-smoke-0_9hmung/fme-tiny-decoder-66wgggj2/result-packet.zip'}


## 설명할 수 있는지 확인하기

1. 마스크를 없애서 next-token 점수가 좋아졌더라도 유효한 개선이 아닌 이유는 무엇인가요?
2. 정확한 재개에는 가중치뿐 아니라 sampler 상태도 필요한 이유가 무엇인가요?
3. 토큰 정답률이 높아도 여러 시퀀스가 틀릴 수 있는 이유는 무엇인가요?

<details><summary>답을 비교해 보세요</summary>
미래 입력은 생성 시점에 알 수 없는 정보를 제공합니다. Sampler 상태는 다음 학습 배치를 보존합니다. 가중치만으로는 옵티마이저 moment, 스케줄과 RNG까지 복구되지 않습니다. 전체 시퀀스 정답률은 토큰 하나라도 틀리면 해당 시퀀스를 오답으로 세며, 자유 생성에서는 오류가 전파될 수도 있습니다.
</details>

## References

- [전체 교재 설명](https://sungeuns.github.io/foundation-model-engineering/ko/chapter-7/tiny-decoder-lab/) — 유도, 마스크, 자원 계산과 확장 질문.
- [PyTorch 재현성 문서](https://docs.pytorch.org/docs/2.15/notes/randomness.html) — 버전·플랫폼·장치 사이의 한계. 2026-11-18 확인.
- [PyTorch 직렬화 문서](https://docs.pytorch.org/docs/2.15/notes/serialization.html) — 상태 사전과 불러오기 규칙. 2026-11-18 확인.
